# Inversion evaluation

Compare camp-A inverters on held-out FlatVel_A maps through the shared
`physics_informed_flow_map.inversion` harness: an `Evaluator` scores any
`InversionModule` (here `FlowTiltModule` and `DiffusionDPSModule`) on the same
held-out targets, same Deepwave operator.

Metrics are reported under three selection rules — `oracle` (lowest MAE, needs the
truth), `gt_free` (lowest data misfit, what a real inversion must use), and
`posterior_mean` — plus `n_solves` (matched-cost PDE-solve count).

In [ ]:
import pandas as pd
import torch
from diffusers import DDPMScheduler

from physics_informed_flow_map.baselines import build_denoiser
from physics_informed_flow_map.flow_matching.models import DiTModelConfig, build_model
from physics_informed_flow_map.inversion import (
    DiffusionDPSModule,
    Evaluator,
    FlowTiltModule,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Prior checkpoints (in the gitignored runs/ — update these to your latest):
FLOW_CKPT = "runs/0001_flow_matching/2026-06-25T14-18-57Z/checkpoints/step_39_ema.pt"
DIFF_CKPT = "runs/0003_baselines/2026-06-26T11-22-31Z/checkpoints/step_99_ema.pt"
N_TARGETS = 8  # held-out FlatVel_A val maps to average over
STEPS, N_SAMPLES = 200, 4

In [ ]:
# Load the two priors (I/O at the edge; the modules just consume them).
flow_prior = build_model(
    (1, 64, 64), None, DiTModelConfig(hidden=256, depth=6, num_heads=8, patch_size=4)
).to(device)
flow_prior.load_state_dict(torch.load(FLOW_CKPT, map_location=device, weights_only=False)["model"])
flow_prior.eval()

denoiser = build_denoiser("unet", sample_size=64, channels=1).to(device)
denoiser.load_state_dict(torch.load(DIFF_CKPT, map_location=device, weights_only=False)["model"])
denoiser.eval()
scheduler = DDPMScheduler(num_train_timesteps=1000)

ev = Evaluator.from_openfwi(["FlatVel_A"], N_TARGETS, device=device)
print(f"{len(ev.targets)} held-out targets")

In [ ]:
# Score each module at its best guidance from the earlier sweep.
modules = [
    FlowTiltModule(flow_prior, guidance=1.0, steps=STEPS, n_samples=N_SAMPLES, device=device),
    DiffusionDPSModule(denoiser, scheduler, guidance=0.3, steps=STEPS, n_samples=N_SAMPLES, device=device),
]

rows = []
for m in modules:
    torch.manual_seed(0)  # same posterior noise across modules for a fair compare
    stats = ev.evaluate(m)
    print(stats, "\n")
    rows.append({"module": m.name, **stats.summary()})

cols = ["module", "mae_gt_free_mean", "mae_oracle_mean", "mae_postmean_mean", "ssim_gt_free_mean", "n_solves"]
pd.DataFrame(rows)[cols].round(3)

## Guidance sweep

MAE (GT-free pick) vs guidance strength for each method on the same held-out set.

In [ ]:
import matplotlib.pyplot as plt

GUIDANCES = [0.3, 1.0, 3.0, 10.0]


def sweep(make_module):
    out = []
    for g in GUIDANCES:
        torch.manual_seed(0)
        out.append(ev.evaluate(make_module(g)).agg["mae_gt_free_mean"])
    return out


flow_mae = sweep(lambda g: FlowTiltModule(flow_prior, guidance=g, steps=STEPS, n_samples=N_SAMPLES, device=device))
diff_mae = sweep(lambda g: DiffusionDPSModule(denoiser, scheduler, guidance=g, steps=STEPS, n_samples=N_SAMPLES, device=device))

plt.figure(figsize=(5, 3.4))
plt.plot(GUIDANCES, flow_mae, "o-", label="flow tilt")
plt.plot(GUIDANCES, diff_mae, "s-", label="diffusion DPS")
plt.xscale("log")
plt.xlabel("guidance strength")
plt.ylabel("MAE (m/s), GT-free pick")
plt.legend()
plt.tight_layout()
plt.show()